# 실습 5-1 · 사진 한 장은 어떤 숫자 배열인가

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

**⏱ 60분** · 5주차 실습 1/2 — 사진을 숫자 배열과 텐서로 다루기 → (5-2) 콘볼루션을 코드로 만들고 작은 CNN을 학습시키기

## 🎯 이번 시간의 질문

지난 네 번의 실습에서 입력은 `(데이터 수, 변수 수)` 모양의 표였습니다. 오늘부터 입력은 **사진**입니다.

> **사진 한 장은 어떤 숫자 배열이고, 신경망에 넣으려면 어떤 모양으로 바꿔야 할까요?**

이 질문에 답하려면 사진을 열어 픽셀 값을 보고, 컬러의 채널을 분해하고, PyTorch가 요구하는 축 순서로 바꾸고, 표처럼 표준화해서 배치로 꺼내야 합니다. 마지막에 이론 5장의 파라미터 폭발을 코드로 확인합니다.

**데이터:** FashionMNIST(28×28 흑백 옷 사진 7만 장), CIFAR-10(32×32 컬러 사진 6만 장). 코드에서 내려받습니다.

**이번 시간에 반복할 패턴:** `np.array(img)`로 배열 만들기 · `.shape`와 슬라이싱으로 픽셀 보기 · `permute`로 축 순서 바꾸기 · `ToTensor`·`Normalize`·`Compose` · `DataLoader`에서 `(N, C, H, W)` 배치 꺼내기.

**대응 이론:** [Ch05 컴퓨터 비전과 이미지 데이터](https://ralbu85.github.io/lecture_deeplearning/chapters/ch05.html)

| 표시 | 뜻 |
|---|---|
| ▶ | 그대로 실행합니다 |
| ✍️ | 보여 준 코드를 손으로 타이핑해서 실행합니다. 빈칸 `___`은 채웁니다 |
| 💬 | 결과를 함께 읽습니다 |
| ✅ | 이 숫자가 나오면 맞습니다 |
| 📝 | 과제. 힌트를 보고 스스로 풉니다 |

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 두 데이터셋을 내려받습니다(처음 한 번은 1~2분).
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

np.set_printoptions(linewidth=140)
torch.manual_seed(42)

fmnist = datasets.FashionMNIST(root='./data', train=True, download=True)
cifar = datasets.CIFAR10(root='./data', train=True, download=True)
print("준비 완료!")

✅ `준비 완료!`가 나오면 됩니다.

---

## 1. 흑백 사진 한 장은 어떤 숫자인가

`fmnist`는 사진과 정답의 쌍을 6만 개 담고 있습니다. 길이와 클래스 이름부터 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(len(fmnist))
print(fmnist.classes)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 60,000장이고 클래스는 티셔츠부터 앵클부츠까지 **10가지**입니다. 각 사진에 클래스 번호가 정답으로 붙어 있습니다.

`fmnist[0]`은 사진 한 장과 정답 번호의 쌍입니다. 사진은 PIL 이미지이고, `np.array`로 숫자 배열이 됩니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
img, label = fmnist[0]
a = np.array(img)

print(label, fmnist.classes[label])
print(a.shape, a.dtype)
print(a.min(), a.max())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 정답은 9번 `Ankle boot`입니다. 배열은 `(28, 28)`이고 값은 **0부터 255까지의 정수**(`uint8`)입니다.
- 이론에서 본 대로 흑백 사진 한 장은 `(높이, 너비)` 행렬이고 칸 하나가 픽셀 하나입니다.
- 0이 검정, 255가 흰색입니다.

가운데 부분의 숫자를 그대로 출력합니다. `a[10:18, 8:20]`은 10~17행, 8~19열입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(a[10:18, 8:20])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 값이 큰 칸(200대)이 밝고 0인 칸이 검습니다. 같은 부분을 그림으로 봅니다.

In [ ]:
# ▶ 그대로 실행하세요 — 전체와 잘라 낸 부분
fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(a, cmap='gray')
axes[0].set_title(fmnist.classes[label])
axes[1].imshow(a[10:18, 8:20], cmap='gray')
axes[1].set_title('a[10:18, 8:20]')
for ax in axes:
    ax.axis('off')
plt.show()

💬 오른쪽이 방금 출력한 숫자 96개입니다. 숫자와 밝기가 일대일로 대응합니다.

---

## 2. 컬러 사진은 채널이 하나 더 있다

CIFAR-10은 32×32 컬러 사진입니다. 한 장을 배열로 만들면 축이 하나 더 있습니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
img_c, label_c = cifar[0]
arr = np.array(img_c)

print(cifar.classes[label_c])
print(arr.shape, arr.dtype)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 개구리(`frog`) 사진이고 배열은 `(32, 32, 3)`입니다. 마지막 축 3이 **R, G, B 채널**입니다. 컬러 사진 한 장은 같은 크기의 흑백 행렬 세 장을 겹친 것입니다.

마지막 축에서 0번을 고르면 빨강 채널 한 장이 나옵니다. `axis=(0, 1)`은 높이와 너비 축을 없애고 **채널마다 평균 하나**를 남깁니다. 실습 1의 축 규칙 그대로입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
red = arr[:, :, 0]
print(red.shape)
print(arr.mean(axis=(0, 1)).round(1))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `(32, 32)`와 `[141.2 105.1  64. ]`이면 맞습니다. 빨강 채널의 평균이 가장 높고 파랑이 가장 낮은 사진입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 원본과 채널 세 장
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(arr)
axes[0].set_title('RGB')
for i, name in enumerate(['R', 'G', 'B']):
    axes[i + 1].imshow(arr[:, :, i], cmap='gray', vmin=0, vmax=255)
    axes[i + 1].set_title(name)
for ax in axes:
    ax.axis('off')
plt.show()

PIL과 matplotlib은 `(높이, 너비, 채널)`로 다루지만, PyTorch는 **채널을 앞에** 두는 `(채널, 높이, 너비)`를 씁니다. `permute`로 축의 순서를 바꿉니다. `permute(2, 0, 1)`은 "원래 2번 축을 0번으로, 0번을 1번으로, 1번을 2번으로" 옮깁니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
t_hwc = torch.tensor(arr)
t_chw = t_hwc.permute(2, 0, 1)
print(tuple(t_hwc.shape), '->', tuple(t_chw.shape))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `(32, 32, 3) -> (3, 32, 32)`이면 맞습니다. 값은 그대로이고 축의 자리만 바뀝니다.

---

## 3. 신경망에 넣기 전 변환

`transforms.ToTensor()`는 PIL 이미지를 텐서로 바꾸면서 두 가지를 한꺼번에 합니다. 축을 `(C, H, W)`로 옮기고, 값을 255로 나누어 **0~1 사이의 실수**로 만듭니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
to_tensor = transforms.ToTensor()
x = to_tensor(img_c)

print(tuple(x.shape), x.dtype)
print(x.min().item(), x.max().item())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — `(3, 32, 32)`, `float32`, 값은 0.0~1.0입니다. 직접 계산해 같은지 대조합니다. `np.allclose`는 두 배열의 값이 거의 같으면 `True`입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
manual = arr.transpose(2, 0, 1) / 255.0
print(np.allclose(x.numpy(), manual))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `True`면 맞습니다. `transpose(2, 0, 1)`은 numpy의 `permute`입니다.

실습 4에서 표의 열마다 표준화했듯이, 이미지는 **채널마다** 표준화합니다. `Normalize(mean, std)`는 채널별로 $(x-\mu)/\sigma$를 계산합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
normalize = transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
xn = normalize(x)

print(xn.min().item(), xn.max().item())
print((x[0, 0, 0].item() - 0.5) / 0.5, xn[0, 0, 0].item())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 0~1이던 값이 **-1~1**로 옮겨졌습니다. 첫 픽셀을 손으로 계산한 값과 `Normalize`의 값이 -0.5373으로 같습니다.

변환 여러 개를 순서대로 적용하는 파이프라인은 `Compose`입니다. `Normalize`는 텐서를 받으므로 `ToTensor` 뒤에 와야 합니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
pipeline = transforms.Compose([
    transforms.___(),                                   # PIL → 텐서, ÷255
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
out = pipeline(img_c)
print(tuple(out.shape), out.min().item(), out.max().item())

✅ **체크포인트** — `(3, 32, 32) -1.0 1.0`이면 맞습니다. 이 파이프라인을 데이터셋에 `transform`으로 넘기면 사진을 꺼낼 때마다 자동으로 적용됩니다.

---

## 4. 배치로 꺼내면 `(N, C, H, W)`

실습 4의 `DataLoader`를 그대로 씁니다. 파이프라인을 붙인 데이터셋에서 64장을 꺼냅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
from torch.utils.data import DataLoader

cifar_t = datasets.CIFAR10(root='./data', train=True, download=False, transform=pipeline)
loader = DataLoader(cifar_t, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print(tuple(xb.shape), tuple(yb.shape))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

✅ **체크포인트** — `(64, 3, 32, 32) (64,)`이면 맞습니다. 앞으로 나오는 모든 이미지 텐서는 이 네 축 **`(N, C, H, W)`**를 갖습니다. N은 배치 크기, C는 채널, H와 W는 높이와 너비입니다.

이론 5장의 방식대로 사진 한 장을 한 줄로 펼쳐 `nn.Linear`에 넣어 봅니다. `flatten(start_dim=1)`은 N축은 두고 나머지를 한 줄로 잇습니다. 파라미터 수는 실습 2의 규칙 $n_{\text{in}} \times n_{\text{out}} + n_{\text{out}}$입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
import torch.nn as nn

flat = xb.flatten(start_dim=1)
fc = nn.Linear(3 * 32 * 32, 128)

print(tuple(flat.shape))
print(fc.weight.shape, fc.bias.shape)
print(3 * 32 * 32 * 128 + 128)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

💬 **결과 해설** — 펼친 길이는 **3,072**이고, 은닉 노드 128개에 연결하면 가중치 `(128, 3072)`에 편향 128개, 합쳐서 **393,344**개입니다.
- 32×32짜리 작은 사진 한 장을 층 하나에 넣는 데 40만 개입니다.
- 사진이 커지면 어떻게 되는지는 과제 2에서 봅니다. 이것이 이론 5장의 **파라미터 폭발**이고, 5-2의 콘볼루션이 푸는 문제입니다.

---

## 5. 실습 과제 (13분)

### 📝 과제 1 · 축 순서를 되돌려 그리기

`(3, 32, 32)`인 `t_chw`를 다시 `(32, 32, 3)`으로 되돌려 원본 `arr`과 같은지 확인하고, `plt.imshow`로 그립니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절의 permute(2, 0, 1)이 (H, W, C)→(C, H, W)였습니다. 되돌리려면 "원래 1번 축을 0번으로, 2번을 1번으로, 0번을 2번으로" 옮기는 permute를 씁니다. 같은지는 np.array_equal(back.numpy(), arr)로 봅니다.

✏️ **나의 답:** `permute(___, ___, ___)`


### 📝 과제 2 · 224×224 사진이면 파라미터는 몇 개인가

224×224 컬러 사진을 같은 은닉 노드 128개에 연결할 때의 파라미터 수를 먼저 손으로 계산한 뒤 코드로 확인합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 4절 마지막 셀에서 32를 224로 바꿉니다. nn.Linear(3 * 224 * 224, 128)의 weight.shape와 공식 3*224*224*128 + 128을 함께 출력합니다.

| 입력 크기 | 펼친 길이 | 파라미터 수 |
|---|---|---|
| 32×32×3 | 3,072 | 393,344 |
| 224×224×3 | ✏️ | ✏️ |


### 📝 과제 3 · 채널 평균으로 사진 고르기

`cifar[1]`의 사진을 배열로 만들어 클래스 이름과 채널별 평균을 출력합니다. `cifar[0]`(개구리)과 비교해 어느 채널이 높은지 봅니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절의 세 줄에서 cifar[0]을 cifar[1]로 바꿉니다. 평균은 .mean(axis=(0, 1)).round(1)입니다.

✏️ **나의 답:** 클래스 ___, 채널 평균 ___


### 📝 과제 4 · 생각해 보기

`Normalize`의 평균과 표준편차를 훈련 사진들로 구한다면, 실습 4의 어떤 규칙을 지켜야 할까요? 한두 문장으로 적습니다.

✏️ **나의 답:**


---

## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 흑백 한 장 | `np.array(img)` → `(28, 28)` uint8, 0~255. 칸 하나가 픽셀 |
| 컬러 한 장 | `(32, 32, 3)`. 마지막 축이 R, G, B. `arr[:, :, 0]`이 빨강 채널 |
| 축 순서 | 보여 줄 때 `(H, W, C)`, PyTorch에 넣을 때 `(C, H, W)`. `permute(2, 0, 1)` |
| 변환 | `ToTensor`(CHW + ÷255) → `Normalize`(채널별 표준화), `Compose`로 묶음 |
| 배치 | `DataLoader`에서 `(N, C, H, W)`. `(64, 3, 32, 32)` |
| 파라미터 폭발 | 펼치면 3,072 → Linear(→128) 393,344개. 224×224면 1,900만 개 |

**다음:** [실습 5-2](https://ralbu85.github.io/lecture_deeplearning/labs/lab05_2.html)에서 콘볼루션을 이중 for문으로 직접 만들어 `F.conv2d`와 대조하고, 파라미터 235개짜리 CNN으로 옷 사진을 분류합니다.